In [ ]:
import json
from pathlib import Path
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, RBF
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR, LinearSVR
from threadpoolctl import threadpool_limits
from xgboost import XGBRegressor

In [ ]:

input_path = Path("derived") / "dataset.xlsx"  # Path to the input workbook.
output_dir = Path("model_comparision_error_bar")  # Folder for generated results.
random_seed = 0  # Random seed for reproducibility.
test_size = 0.20  # Test size.
n_splits = 5  # Number of cross-validation folds.
fold_jobs = 5  # Fold jobs.
dpi = 600  # Resolution in dots per inch.
calc_columns = ["Calc T", "Calc D"]  # Calcination temperature and duration columns.
inner_splits = 3  # Inner splits.
grouped_trials = 8  # Grouped trials.
grouped_alpha_count = 15  # Grouped alpha count.
random_alpha_count = 100  # Random alpha count.
random_trials = {  # Random trials.
    "SVR-linear": 100,
    "SVR-kernel": 100,
    "Gaussian Process Regression": 50,
    "Random Forest": 100,
    "XGBoost": 100,
}

optuna.logging.set_verbosity(optuna.logging.WARNING)


In [ ]:
features = [  
    "wt% Cu",
    "wt% Support1", "χ Support1",
    "wt% Support2", "χ Support2",
    "wt% Support3", "χ Support3",
    "wt% Promoter", "χ Promoter",
    "Synthesis Method",
    "Calc T", "Calc D",
    "H2/CO2", "GHSV", "P", "T",
    "SBET", "SACu", "dCu",
]
target = "STY"  
point_id = "Data point number"  # Unique experimental-record identifier column.

data = pd.read_excel(input_path, sheet_name="data", engine="openpyxl")  # Input records for this analysis.
group_fields = {"Catalyst": "Catalyst number", "Publication": "Reference DOI"}  
flag_columns = {column: f"{column} imputed" for column in calc_columns}  # Columns indicating missing calcination values. Current column name.
required = [point_id, *features, target, *group_fields.values(), *flag_columns.values()]  
missing = set(required) - set(data.columns)  
if missing:
    raise ValueError(f"Missing columns: {sorted(missing)}")

values = data[features + [target]].apply(pd.to_numeric)  # Values used in the current calculation.
if not np.isfinite(values.to_numpy(dtype=float)).all():
    raise ValueError("The encoded model features and STY must contain finite numeric values.")
for column, flag_column in flag_columns.items():  # Current column name; Flag column.
    flags = data[flag_column]  
    if not flags.isin([True, False]).all():
        raise ValueError(f"Expected Boolean flags for {column}.")
    values[column] = values[column].mask(flags.astype(bool))
if data[point_id].isna().any() or not data[point_id].is_unique:
    raise ValueError("Data point numbers must be present and unique.")

X_train, X_test, y_train, y_test = train_test_split(  # Development-set predictors; Held-out test predictors; Development-set responses; Held-out test responses.
    values[features], values[target], test_size=test_size, random_state=random_seed,
)
folds = list(KFold(n_splits=n_splits, shuffle=True, random_state=random_seed).split(X_train))  # Cross-validation splits.
assignments = data[[point_id]].assign(Set="Test")  
assignments["CV validation fold"] = pd.Series(pd.NA, index=data.index, dtype="Int64")
assignments.loc[X_train.index, "Set"] = "Train"
for fold_number, (_, validation) in enumerate(folds, start=1):  # Current cross-validation fold number; Validation.
    assignments.loc[X_train.index[validation], "CV validation fold"] = fold_number

output_dir.mkdir(exist_ok=True)
print(f"Records: {len(data)} | Train: {len(X_train)} | Test: {len(X_test)}")
print("CV validation fold sizes:", [len(validation) for _, validation in folds])  # Validation.


In [ ]:
def scaled(estimator):  # Put numerical standardization and the estimator in one pipeline.
    return make_pipeline(StandardScaler(), estimator)


def gpr_model(kernel_type, alpha, n_restarts_optimizer, random_state):  # Build the standardized Gaussian process model and kernel.
    bounds = (1e-3, 1e3)  
    kernel = (RBF(1.0, bounds) if kernel_type == "RBF"  # Kernel.
              else Matern(1.0, bounds, nu=float(kernel_type.split("_")[1])))
    return scaled(GaussianProcessRegressor(
        kernel=ConstantKernel(1.0, bounds) * kernel, alpha=alpha, normalize_y=True,
        n_restarts_optimizer=n_restarts_optimizer, random_state=random_state,
    ))


def search_parameters(trial, name, y_outer):  # Draw candidate hyperparameters from the specified search space.
    y_std = float(y_outer.std(ddof=0))  # Y standard deviation.
    if name == "SVR-linear":
        return {
            "C": trial.suggest_float("C", 1e-3, 1e5, log=True),
            "epsilon": trial.suggest_float("epsilon", y_std * 0.0001, y_std * 0.5),
        }
    if name == "SVR-kernel":
        return {
            "C": trial.suggest_float("C", 1e-2, 1e5, log=True),
            "gamma": trial.suggest_float("gamma", 1e-5, 1e1, log=True),
            "epsilon": trial.suggest_float("epsilon", y_std * 0.01, y_std * 0.2),
        }
    if name == "Gaussian Process Regression":
        return {
            "kernel_type": trial.suggest_categorical("kernel_type", ["RBF", "Matern_1.5", "Matern_2.5"]),
            "alpha": trial.suggest_float("alpha", 1e-4, 10, log=True),
        }
    if name == "Random Forest":
        return {
            "max_depth": trial.suggest_int("max_depth", 3, 15),
            "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 15),
            "max_features": trial.suggest_categorical("max_features", [1.0, "sqrt", "log2"]),
        }
    return {
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 10, 30),
        "subsample": trial.suggest_float("subsample", 0.5, 0.85),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.85),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-2, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 50, log=True),
        "gamma": trial.suggest_float("gamma", 0, 5),
    }


model_names = [  
    "Linear Regression", "Ridge", "Lasso", "SVR-linear", "SVR-kernel",
    "Gaussian Process Regression", "Random Forest", "XGBoost",
]


def build_model(name, params, final=False):  # Construct the requested regressor from the selected hyperparameters.
    if name == "Linear Regression":
        return scaled(LinearRegression())
    if name == "Ridge":
        return scaled(Ridge(**params))
    if name == "Lasso":
        return scaled(Lasso(**params, max_iter=10000, random_state=random_seed))
    if name == "SVR-linear":
        return scaled(LinearSVR(
            **params, dual=True, loss="epsilon_insensitive",
            max_iter=300000 if final else 100000, random_state=random_seed,
        ))
    if name == "SVR-kernel":
        return scaled(SVR(kernel="rbf", **params))
    if name == "Gaussian Process Regression":
        return gpr_model(**params, n_restarts_optimizer=5 if final else 0,
                         random_state=random_seed if final else 42)
    if name == "Random Forest":
        return RandomForestRegressor(
            **params, n_estimators=300, random_state=42 if final else random_seed, n_jobs=1,
        )
    settings = {} if final else {"n_estimators": 1000, "early_stopping_rounds": 20}  # Settings.
    return XGBRegressor(
        **params, **settings, objective="reg:squarederror", tree_method="hist",
        eval_metric="rmse", random_state=random_seed, n_jobs=1,
    )


In [ ]:
metric_names = ["R2", "RMSE", "MAE"]  # Metric names.
imputation_records = {}  # Imputation records.


def impute_calcination(X_fit, X_eval, scheme, fold=None, outer_fold=None):  # Fit rounded calcination means on the training partition and apply them to both partitions.
    means = X_fit[calc_columns].mean()  # Calculated means for imputation or plotting.
    if means.isna().any():
        raise ValueError(f"{scheme} fold {fold}: a training column lacks observed calcination data.")
    fill = np.floor(means + 0.5)  # Rounded imputation values from the fitting data.
    for column in calc_columns:  # Current column name.
        imputation_records[scheme, outer_fold, fold, column] = {
            "Scheme": scheme, "Outer fold": outer_fold, "Fold": fold, "Feature": column,
            "Train records": len(X_fit), "Evaluation records": len(X_eval),
            "Observed train records": int(X_fit[column].count()),
            "Train mean": means[column], "Fill value": int(fill[column]),
            "Train missing": int(X_fit[column].isna().sum()),
            "Evaluation missing": int(X_eval[column].isna().sum()),
        }
    return X_fit.fillna(fill), X_eval.fillna(fill)


def make_splits(X, y, indices, scheme, outer_fold=None):  # Create random or grouped cross-validation partitions.
    splits = []  # Splits.
    for fold_number, (train_rows, eval_rows) in enumerate(indices, start=1):  # Current cross-validation fold number; Train rows; Eval rows.
        X_fit, X_eval = impute_calcination(  # Predictors used to fit the estimator; Evaluation predictors.
            X.iloc[train_rows], X.iloc[eval_rows], scheme, fold_number, outer_fold,
        )
        splits.append((X_fit, y.iloc[train_rows], X_eval, y.iloc[eval_rows]))
    return splits


def regression_metrics(actual, predicted, clip_predictions=True):  # Compute R-squared, RMSE, and MAE for a regression partition.
    if clip_predictions:
        predicted = np.clip(predicted, 0, None)
    return {
        "R2": float(r2_score(actual, predicted)),
        "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
        "MAE": float(mean_absolute_error(actual, predicted)),
    }


def fit_metrics(estimator, split, early_stopping=False, clip_predictions=True):  # Fit a candidate model and score the evaluation partition.
    X_fit, y_fit, X_eval, y_eval = split  # Predictors used to fit the estimator; Responses used to fit the estimator; Evaluation predictors; Y eval.
    model = clone(estimator)  # Estimator for the current task.
    if early_stopping:
        model.fit(X_fit, y_fit, eval_set=[(X_eval, y_eval)], verbose=False)
    else:
        model.fit(X_fit, y_fit)
    result = regression_metrics(y_eval, model.predict(X_eval), clip_predictions)  # Result.
    if early_stopping:
        result["Best boosting rounds"] = model.best_iteration + 1
    return result


def score_splits(estimator, splits, early_stopping=False, clip_predictions=True):  # Fit and score the estimator on each supplied split.
    with threadpool_limits(limits=1):
        return Parallel(n_jobs=fold_jobs, prefer="threads")(
            delayed(fit_metrics)(estimator, split, early_stopping, clip_predictions)
            for split in splits  # Training and held-out partition assignments.
        )


def summarize_folds(scores):  # Summarize validation metrics across folds.
    values = pd.DataFrame(scores)  # Values used in the current calculation.
    return {
        key: statistic
        for metric in metric_names  # Metric.
        for key, statistic in (  # Key; Statistic.
            (f"Mean CV {metric}", values[metric].mean()),
            (f"CV {metric} SD", values[metric].std(ddof=1)),
        )
    }


def select_model(name, splits, y_tuning, grouped=False):  # Select model hyperparameters using only the supplied training data.
    if name == "Linear Regression":
        return build_model(name, {}), {}, None

    if name in ("Ridge", "Lasso"):
        alpha_count = grouped_alpha_count if grouped else random_alpha_count  # Alpha count.
        upper_exponent = 4 if name == "Ridge" else 1  # Upper exponent.
        candidates = np.logspace(-4, upper_exponent, alpha_count)  # Candidates.
        best_value = np.inf  # Best value.
        for alpha in candidates:  
            scores = score_splits(build_model(name, {"alpha": alpha}), splits, clip_predictions=False)  # Scores for the current evaluation.
            rmse = np.asarray([fold["RMSE"] for fold in scores])  # RMSE. Cross-validation fold.
            value = np.mean(rmse ** 2) if name == "Lasso" else np.mean(rmse)  # Current value.
            if value < best_value:
                best_alpha, best_value = alpha, value  # Best alpha.
        params = {"alpha": float(best_alpha)}  # Estimator hyperparameters.
        return build_model(name, params), params, None

    def objective(trial):  # Return cross-validation error for the current Optuna trial.
        params = search_parameters(trial, name, y_tuning)  # Estimator hyperparameters.
        scores = score_splits(  # Scores for the current evaluation.
            build_model(name, params), splits,
            early_stopping=name == "XGBoost", clip_predictions=name != "SVR-linear",
        )
        if name == "XGBoost":
            trial.set_user_attr("fold_scores", scores)
        return float(np.mean([fold["RMSE"] for fold in scores]))  # Cross-validation fold.

    study = optuna.create_study(  # Optuna hyperparameter search.
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=random_seed, n_startup_trials=4 if grouped else 10),
    )
    study.optimize(objective, n_trials=grouped_trials if grouped else random_trials[name])
    params = study.best_params
    scores = None
    if name == "XGBoost":
        scores = study.best_trial.user_attrs["fold_scores"]
        mean_rounds = int(np.mean([fold["Best boosting rounds"] for fold in scores]))  # Mean rounds. Cross-validation fold.
        params["n_estimators"] = max(1, int(1.1 * mean_rounds))
    return build_model(name, params, final=True), params, scores


In [ ]:
random_splits = make_splits(X_train, y_train, folds, "Random CV")  
X_train_filled, X_test_filled = impute_calcination(X_train, X_test, "Final holdout")  


In [ ]:


correlations = X_train_filled.corr(method="pearson")  
correlation_labels = {  
    "H2/CO2": r"H$_2$/CO$_2$",
    "SBET": r"S$_{\mathrm{BET}}$",
    "SACu": r"SA$_{\mathrm{Cu}}$",
    "dCu": r"d$_{\mathrm{Cu}}$",
}
corr_plot = correlations.rename(index=correlation_labels, columns=correlation_labels)  
lower_mask = np.triu(np.ones(corr_plot.shape, dtype=bool), k=1)  
annotations = corr_plot.round(2).astype(str).mask(lower_mask, "")  

with mpl.rc_context({
    "font.family": ["Times New Roman", "DejaVu Serif"],
    "font.weight": "bold",
    "mathtext.default": "regular",
    "axes.unicode_minus": False,
}):
    figure, ax = plt.subplots(figsize=(10, 8))  # Matplotlib figure; Plot axes.
    sns.heatmap(
        corr_plot, annot=annotations, fmt="", annot_kws={"size": 8, "weight": "bold"},
        cmap="RdYlBu", vmin=-1, vmax=1, center=0, square=True,
        cbar_kws={"shrink": 0.85, "label": "Pearson correlation", "ticks": np.linspace(-1, 1, 9)},
        linewidths=0.5, linecolor="white", ax=ax,
    )
    ax.tick_params(labelsize=12)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
    colorbar = ax.collections[0].colorbar  # Colorbar.
    colorbar.ax.tick_params(labelsize=11)
    colorbar.set_label("Pearson correlation", fontsize=13, weight="bold")
    figure.savefig(output_dir / "Heatmap_Pearson.tif", dpi=dpi, bbox_inches="tight",  # Save Fig. 3, the training-set Pearson correlation matrix.
                   pil_kwargs={"compression": "tiff_lzw"})
    plt.show()
    plt.close(figure)

In [ ]:
random_rows, random_fold_rows = [], []
for name in model_names:  # Current item name.
    print(f"Training {name}...", flush=True)
    final_model, params, scores = select_model(name, random_splits, y_train)  # Final model; Estimator hyperparameters; Scores for the current evaluation.
    if scores is None:
        scores = score_splits(final_model, random_splits)

    with threadpool_limits(limits=1):
        final_model.fit(X_train_filled, y_train)
        train = regression_metrics(y_train, final_model.predict(X_train_filled))  # Development records or their indices.
        test = regression_metrics(y_test, final_model.predict(X_test_filled))  

    row = {"Model": name, **summarize_folds(scores)} 
    for metric in metric_names:  
        row[f"Train {metric}"], row[f"Test {metric}"] = train[metric], test[metric]
    row["Parameters"] = json.dumps(params, sort_keys=True)
    random_rows.append(row)
    random_fold_rows.extend({"Model": name, "Fold": i, **s} for i, s in enumerate(scores, start=1))  
    print(f"  Mean CV R2: {row['Mean CV R2']:.4f} | Test R2: {test['R2']:.4f}", flush=True)

random_summary = pd.DataFrame(random_rows).set_index("Model")  # Random summary.
random_folds = pd.DataFrame(random_fold_rows)  # Random folds.
display(random_summary.drop(columns="Parameters").round(4))


In [ ]:
grouped_fold_rows = []  


for scheme, group_column in group_fields.items():  
    groups = data.loc[X_train.index, group_column]  
    if groups.isna().any():
        raise ValueError(f"Missing group labels in {group_column}.")
    if scheme == "Publication":
        groups = groups.astype("string").str.strip().str.lower()
        if groups.eq("").any():
            raise ValueError("Publication DOI labels must be present.")

    outer_indices = list(GroupKFold(n_splits=n_splits).split(X_train, y_train, groups))  
    outer_splits = make_splits(X_train, y_train, outer_indices, f"{scheme} outer CV")  
    assignments[f"{scheme} CV fold"] = pd.Series(pd.NA, index=data.index, dtype="Int64")
    for fold_number, ((train_rows, eval_rows), prepared) in enumerate(zip(outer_indices, outer_splits), start=1):  
        train_groups, eval_groups = groups.iloc[train_rows], groups.iloc[eval_rows]  
        if set(train_groups) & set(eval_groups):
            raise ValueError(f"{scheme} groups overlap in fold {fold_number}.")
        assignments.loc[X_train.index[eval_rows], f"{scheme} CV fold"] = fold_number
        X_outer, y_outer = X_train.iloc[train_rows], y_train.iloc[train_rows]  
        print(f"{scheme} fold {fold_number}/{n_splits} | Validation records: {len(eval_rows)}", flush=True)

        inner_indices = GroupKFold(n_splits=inner_splits).split(X_outer, y_outer, train_groups)  
        inner = make_splits(X_outer, y_outer, inner_indices, f"{scheme} inner CV", fold_number)  
        for name in model_names:
            model, params, _ = select_model(name, inner, y_outer, grouped=True)  # Estimator for the current task.
            with threadpool_limits(limits=1):
                scores = fit_metrics(model, prepared)
            grouped_fold_rows.append({
                "Scheme": scheme, "Model": name, "Fold": fold_number,
                "Train records": len(train_rows), "Validation records": len(eval_rows),
                "Train groups": train_groups.nunique(), "Validation groups": eval_groups.nunique(),
                **scores, "Parameters": json.dumps(params, sort_keys=True),
            })

grouped_folds = pd.DataFrame(grouped_fold_rows)  # Grouped folds.
grouped_summary = pd.DataFrame([  # Grouped summary.
    {"Scheme": scheme, "Model": model, **summarize_folds(scores.to_dict("records"))}
    for (scheme, model), scores in grouped_folds.groupby(["Scheme", "Model"], sort=False)  # Scheme; Estimator for the current task; Scores for the current evaluation.
])
display(grouped_summary.round(4))


In [ ]:
comparison_style = {  
    "font.family": ["Arial", "DejaVu Sans"],
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": 1.2,
}


def draw_comparison_with_error_bars(ax, metric, ylabel):  # Draw cross-validation and final scores for one metric.
    styles = [("Mean CV", "#666666"), ("Train", "#C32628"), ("Test", "#2945C2")] 
    positions = np.arange(len(random_summary))  
    bar_width = 0.27  # Bar width.
    metric_label = r"$R^2$" if metric == "R2" else metric  
    spread = random_summary[f"CV {metric} SD"].to_numpy() 
    means = random_summary[f"Mean CV {metric}"].to_numpy()  # Calculated means for imputation or plotting.
    scores = random_summary[[f"{group} {metric}" for group, _ in styles]].to_numpy()  # Scores for the current evaluation. Group.
    low = min(0.0, float(scores.min()), float((means - spread).min()))  # Lower bound for the displayed values.
    high = max(float(scores.max()), float((means + spread).max()))  # Upper bound for the displayed values.
    span = max(high - low, 1e-8)  # Span.
    label_offset = 0.012 * span  # Label offset.

    legend = []  
    gradient = np.linspace(0, 1, 256).reshape(-1, 1)  

    for offset, (group, color) in enumerate(styles, start=-1):  # Offset; Group; Color assigned to the current series.
        values = random_summary[f"{group} {metric}"].to_numpy()  # Values used in the current calculation.
        errors = spread if group == "Mean CV" else np.zeros_like(values)  
        centers = positions + offset * bar_width  # Centers.
        bars = ax.bar(  # Bars.
            centers, values, width=bar_width,
            facecolor="none", edgecolor="black", linewidth=1.1, zorder=2,
        )
        cmap = LinearSegmentedColormap.from_list(f"{metric}_{group}", ["black", color])  # Matplotlib color map.
        for bar, value, error in zip(bars, values, errors):  # Bar; Current value; Error.
            x, width = bar.get_x(), bar.get_width()  # Horizontal coordinate or predictor values; Width.
            if value != 0:
                image = ax.imshow(  
                    gradient, extent=(x, x + width, min(0, value), max(0, value)),
                    cmap=cmap, origin="lower", aspect="auto", zorder=1,
                )
                image.set_clip_path(bar)
            ax.text(
                x + width / 2, value + error + label_offset, f"{value:.2f}",
                ha="center", va="bottom", fontsize=9 if metric == "R2" else 8,
                weight="bold", zorder=4,
            )
        if group == "Mean CV":
            ax.errorbar(centers, values, yerr=errors, fmt="none", ecolor="black",
                        elinewidth=1.2, capsize=3, capthick=1.2, zorder=3)
        suffix = " ± SD" if group == "Mean CV" else ""  
        legend.append(Patch(facecolor=color, edgecolor="black",
                            label=f"{group} {metric_label}{suffix}"))

    ax.set_xlim(-0.76, len(random_summary) - 0.24)
    ax.set_ylim(low - (0.12 * span if low < 0 else 0), high + 0.17 * span)
    ax.yaxis.set_major_locator(MultipleLocator(0.2 if metric == "R2" else 20))
    ax.set_xticks(positions, random_summary.index, rotation=15, ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Model Evaluation by {metric_label}", pad=12)
    ax.legend(handles=legend, loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)

def save_comparison(figure, filename):  # Save and display a model comparison panel.
    figure.savefig(output_dir / filename, dpi=dpi, bbox_inches="tight",
                   pil_kwargs={"compression": "tiff_lzw"})
    plt.show()
    plt.close(figure)


unit = r"mg$_{\mathrm{MeOH}}$ h$^{-1}$ g$_{\mathrm{cat}}^{-1}$"  
comparison_plots = [  # Comparison plots.
    ("R2", r"$R^2$ Score", "Model_Comparison_R2_error_bar.tif"),  # Fig. 4, candidate-model R-squared with cross-validation error bars.
    ("RMSE", f"RMSE ({unit})", "Model_Comparison_RMSE_error_bar.tif"),  # Fig. S1a, candidate-model RMSE.
    ("MAE", f"MAE ({unit})", "Model_Comparison_MAE_error_bar.tif"),  # Fig. S1b, candidate-model MAE.
]

with mpl.rc_context(comparison_style):
    for metric, ylabel, filename in comparison_plots: 
        figure, ax = plt.subplots(figsize=(12 if metric == "R2" else 13, 5.5))
        draw_comparison_with_error_bars(ax, metric, ylabel)
        figure.tight_layout()
        save_comparison(figure, filename)  # Save Fig. 4, Fig. S1a, or Fig. S1b for the selected metric.

    figure, axes = plt.subplots(2, 1, figsize=(13, 9.6), layout="constrained")  # Plot axes arranged by panel.
    for label, ax, (metric, ylabel, _) in zip(("(a)", "(b)"), axes, comparison_plots[1:]):  
        draw_comparison_with_error_bars(ax, metric, ylabel)
        ax.annotate(
            label, (0, 1), xycoords="axes fraction", xytext=(-18, 8),
            textcoords="offset points", ha="left", va="bottom", fontsize=16,
            fontweight="bold", annotation_clip=False,
        )
    save_comparison(figure, "Model_Comparison_RMSE_MAE_error_bar.tif")  # Save Fig. S1a (RMSE) and Fig. S1b (MAE) as a combined figure.


In [ ]:
validation_notes = pd.DataFrame([  # Validation notes.
    ("Random CV", "Five shuffled folds within the training set. Hyperparameters were selected using these folds."),
    ("Random error bars", "Mean ± sample standard deviation across the five validation folds (ddof=1)."),
    ("Holdout evaluation", "Final models and imputation values were fitted on the full training set and applied to the held-out test set."),
    ("Calcination imputation", "Flagged cells were restored to missing in memory. For each split, both partitions used rounded means of observed values from its training partition."),
    ("Grouped scope", f"Both group-based evaluations use the same {len(X_train)} records as the original random CV."),
    ("Catalyst grouping", "GroupKFold on the catalyst numbers as recorded in the encoded dataset."),
    ("Publication grouping", "GroupKFold on stripped, lowercased reference DOIs."),
    ("Grouped tuning", f"{inner_splits} inner group folds per outer fold, {grouped_trials} Optuna trials or {grouped_alpha_count} alpha candidates."),
    ("Grouped XGBoost", "Boosting rounds for outer validation were selected using inner folds only."),
    ("STY error units", "mg MeOH h^-1 gcat^-1"),
], columns=["Setting", "Description"])

report_path = output_dir / "model_comparision_error_bar.xlsx"  # Report path.
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:  # Excel writer for the result workbook.
    random_summary.to_excel(writer, sheet_name="summary")
    random_folds.to_excel(writer, sheet_name="cv_folds", index=False)
    grouped_summary.to_excel(writer, sheet_name="grouped_summary", index=False)
    grouped_folds.to_excel(writer, sheet_name="grouped_folds", index=False)
    assignments.to_excel(writer, sheet_name="split", index=False)
    validation_notes.to_excel(writer, sheet_name="validation_notes", index=False)
    pd.DataFrame(imputation_records.values()).to_excel(writer, sheet_name="imputation", index=False)

print(f"Saved figures and results to: {output_dir.resolve()}")
